# BirdCLEF+ 2025 — Exploratory Data Analysis

Goal: understand the data before modelling, and produce figures/numbers for the report.

Sections
1. Files and metadata
2. Class distribution (imbalance)
3. Taxonomic groups (birds / amphibians / mammals / insects)
4. Data sources, quality ratings, secondary labels
5. Recording durations
6. Listening & spectrograms
7. Unlabelled soundscapes and the test format
8. Recording locations
9. Summary of findings

Figures are saved to `figures/` (in Kaggle's output) so they can be dropped into the report.

In [ ]:
import os, glob, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display
from IPython.display import Audio, display

SEED = 42
random.seed(SEED); np.random.seed(SEED)

def find_data_dir():
    # On Kaggle the competition is mounted under /kaggle/input; locally we use data/
    hits = glob.glob("/kaggle/input/**/birdclef-2025/train.csv", recursive=True) \
        or glob.glob("/kaggle/input/*/train.csv")
    if hits:
        return Path(hits[0]).parent
    return Path("../../data")

DATA = find_data_dir()
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
SR = 32_000  # competition audio sample rate

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

print("Data dir:", DATA)
print(sorted(os.listdir(DATA)))

## 1. Files and metadata

In [ ]:
train = pd.read_csv(DATA / "train.csv")
taxonomy = pd.read_csv(DATA / "taxonomy.csv")
sample_sub = pd.read_csv(DATA / "sample_submission.csv")

print("train.csv:", train.shape)
print("taxonomy.csv:", taxonomy.shape)
print("sample_submission.csv:", sample_sub.shape)
display(train.head())
display(taxonomy.head())

In [ ]:
train.info()
print()
print("Missing values per column:")
print(train.isna().sum()[train.isna().sum() > 0])

In [ ]:
for sub in ["train_audio", "train_soundscapes", "test_soundscapes"]:
    p = DATA / sub
    n = len(list(p.rglob("*.ogg"))) if p.exists() else 0
    print(f"{sub:20s} {n:6d} .ogg files")
print()
print((DATA / "recording_location.txt").read_text())

## 2. Class distribution

How many training recordings does each species have? Heavy imbalance means rare species are hard to learn and matter equally for the macro-averaged ROC-AUC metric.

In [ ]:
counts = train["primary_label"].value_counts()
n_classes = taxonomy["primary_label"].nunique()
print(f"Species in taxonomy: {n_classes}")
print(f"Species with >=1 training clip: {counts.size}")
print(f"Species with NO training clip: {n_classes - counts.size}")
print()
print(counts.describe().round(1))
for t in [1, 5, 10, 20, 50]:
    print(f"species with <= {t:3d} clips: {(counts <= t).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(counts)), counts.values, width=1.0)
ax.set_yscale("log")
ax.set_xlabel("Species (sorted by number of clips)")
ax.set_ylabel("Training clips (log scale)")
ax.set_title("Class imbalance in train_audio")
savefig("class_imbalance")

## 3. Taxonomic groups

In [ ]:
tax = taxonomy.set_index("primary_label")
train["class_name"] = train["primary_label"].map(tax["class_name"])

group = pd.DataFrame({
    "species": taxonomy["class_name"].value_counts(),
    "clips": train["class_name"].value_counts(),
}).fillna(0).astype(int)
group["clips_per_species"] = (group["clips"] / group["species"]).round(1)
display(group)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
group["species"].plot.bar(ax=axes[0], title="Species per group")
group["clips"].plot.bar(ax=axes[1], title="Training clips per group", logy=True)
for a in axes: a.set_xlabel("")
savefig("taxonomic_groups")

In [ ]:
# The rarest species overall — candidates for extra data / augmentation
rare = counts[counts <= 5].rename("clips").to_frame()
rare = rare.join(tax[["common_name", "class_name"]])
display(rare.sort_values("clips"))

## 4. Data sources, quality ratings, secondary labels

In [ ]:
if "collection" in train.columns:
    display(train["collection"].value_counts().rename("clips").to_frame())

if "rating" in train.columns:
    fig, ax = plt.subplots(figsize=(6, 3))
    train["rating"].value_counts().sort_index().plot.bar(ax=ax)
    ax.set_xlabel("Rating (0 = unrated)"); ax.set_ylabel("Clips")
    ax.set_title("Recording quality ratings")
    savefig("ratings")

In [ ]:
import ast
sec = (train["secondary_labels"].fillna("[]").apply(ast.literal_eval)
       .apply(lambda xs: [x for x in xs if x]))  # drop empty strings like ['']
has_sec = sec.str.len() > 0
print(f"Clips with secondary labels: {has_sec.sum()} / {len(train)} ({has_sec.mean():.1%})")
print("Mean secondary labels per clip (when present):", round(sec[has_sec].str.len().mean(), 2))

## 5. Recording durations

Read header info from a random sample of files (fast; no decoding).

In [ ]:
N_SAMPLE = min(4000, len(train))
sample = train.sample(N_SAMPLE, random_state=SEED).copy()

def info(fname):
    i = sf.info(DATA / "train_audio" / fname)
    return pd.Series({"duration": i.duration, "sr": i.samplerate, "channels": i.channels})

sample = sample.join(sample["filename"].apply(info))
print("Sample rates:", sample["sr"].value_counts().to_dict())
print("Channels:", sample["channels"].value_counts().to_dict())
display(sample["duration"].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(1))
print(f"Shorter than 5 s: {(sample['duration'] < 5).mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(sample["duration"].clip(upper=120), bins=60)
ax.axvline(5, color="red", ls="--", label="5 s prediction window")
ax.set_xlabel("Duration (s, clipped at 120)"); ax.set_ylabel("Clips")
ax.set_title(f"Training clip durations (sample of {N_SAMPLE})")
ax.legend()
savefig("durations")

## 6. Listening & spectrograms

One example per taxonomic group. The mel spectrogram is the image-like input most BirdCLEF models use.

In [ ]:
def mel_db(y, sr=SR, n_mels=128, fmin=50, fmax=14_000):
    m = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512,
                                       n_mels=n_mels, fmin=fmin, fmax=fmax)
    return librosa.power_to_db(m, ref=np.max)

examples = (train[train["class_name"].notna()]
            .sort_values("rating", ascending=False)
            .groupby("class_name").head(1))

fig, axes = plt.subplots(len(examples), 2, figsize=(12, 2.6 * len(examples)))
axes = np.atleast_2d(axes)
for row, (_, r) in zip(axes, examples.iterrows()):
    y, _ = librosa.load(DATA / "train_audio" / r["filename"], sr=SR, duration=10)
    librosa.display.waveshow(y, sr=SR, ax=row[0])
    row[0].set_title(f"{r['class_name']}: {r['common_name']}")
    img = librosa.display.specshow(mel_db(y), sr=SR, hop_length=512, x_axis="time",
                                   y_axis="mel", fmin=50, fmax=14_000, ax=row[1])
    row[1].set_title("Mel spectrogram (dB)")
savefig("example_spectrograms")

for _, r in examples.iterrows():
    print(r["class_name"], "-", r["common_name"])
    y, _ = librosa.load(DATA / "train_audio" / r["filename"], sr=SR, duration=10)
    display(Audio(y, rate=SR))

## 7. Unlabelled soundscapes and the test format

Test data are 1-minute soundscapes; we must predict a probability for every species in each 5-second window.

In [ ]:
scapes = sorted((DATA / "train_soundscapes").glob("*.ogg"))
print("Unlabelled train soundscapes:", len(scapes))
if scapes:
    durs = pd.Series([sf.info(p).duration for p in scapes[:300]])
    print("Durations of first 300 (s):"); print(durs.describe().round(1))

    y, _ = librosa.load(scapes[0], sr=SR)
    fig, ax = plt.subplots(figsize=(12, 3))
    librosa.display.specshow(mel_db(y), sr=SR, hop_length=512, x_axis="time",
                             y_axis="mel", fmin=50, fmax=14_000, ax=ax)
    for t in range(5, int(len(y) / SR), 5):
        ax.axvline(t, color="white", lw=0.6, alpha=0.7)
    ax.set_title(f"Soundscape {scapes[0].name} with 5 s windows")
    savefig("soundscape_windows")

In [ ]:
print((DATA / "test_soundscapes" / "readme.txt").read_text())
print("sample_submission columns:", sample_sub.shape[1], "(row_id +", sample_sub.shape[1] - 1, "species)")
display(sample_sub.iloc[:5, :6])
missing_in_train = set(sample_sub.columns[1:]) - set(counts.index)
print("Species in submission with no training clip:", len(missing_in_train))

## 8. Recording locations

In [ ]:
if {"latitude", "longitude"}.issubset(train.columns):
    loc = train.dropna(subset=["latitude", "longitude"])
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, g in loc.groupby("class_name"):
        ax.scatter(g["longitude"], g["latitude"], s=3, alpha=0.4, label=name)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title("Where training recordings were made")
    ax.legend(markerscale=4)
    savefig("locations")

## 9. Summary of findings

Key numbers to quote in the report (re-run to refresh).

In [ ]:
summary = {
    "train clips": len(train),
    "species (taxonomy)": n_classes,
    "species with no training clip": n_classes - counts.size,
    "species with <=5 clips": int((counts <= 5).sum()),
    "max / median clips per species": f"{counts.max()} / {int(counts.median())}",
    "clips with secondary labels": f"{has_sec.mean():.1%}",
    "median clip duration (s)": round(sample["duration"].median(), 1),
    "unlabelled soundscapes": len(scapes),
}
summary = pd.Series(summary, name="value").to_frame()
summary.to_csv(FIG / "eda_summary.csv")
display(summary)

**Notes for the report (fill in after reading the outputs):**
- Imbalance: …
- Weak labels — a clip is labelled for the whole file, but the species may only call for part of it: …
- Domain shift — training clips are focused recordings; test data are noisy soundscapes: …
- Implications for our method: …